# Grad-CAM visualisations

In [ ]:
from PIL import Image
import clip, torch
from torchray.attribution.common import Probe, get_module
from torchray.attribution.grad_cam import gradient_to_grad_cam_saliency
import torch.nn.functional as F
import os
from torchvision.utils import save_image
from torchvision import transforms
import matplotlib.pyplot as plt
import cv2
import numpy as np

In [ ]:
fish_classes = [
            'Bangus', 'Big Head Carp', 'Black Spotted Barb', 'Catfish', 'Climbing Perch', 'Fourfinger Threadfin', 'Freshwater Eel', 'Glass Perchlet', 'Goby', 'Gold Fish', 'Gourami', 'Grass Carp', 'Green Spotted Puffer', 'Indian Carp', 'Indo-Pacific Tarpon', 'Jaguar Gapote', 'Janitor Fish', 'Knifefish', 'Long-Snouted Pipefish', 'Mosquito Fish', 'Mudfish', 'Mullet', 'Pangasius', 'Perch', 'Scat Fish', 'Silver Barb', 'Silver Carp','Silver Perch', 'Snakehead', 'Tenpounder', 'Tilapia'
        ]

Helpers

In [ ]:
# Reference: https://github.com/facebookresearch/TorchRay/blob/master/torchray/attribution/common.py

def resize_saliency(tensor, saliency, size, mode):
    """Resize a saliency map.

    Args:
        tensor (:class:`torch.Tensor`): reference tensor.
        saliency (:class:`torch.Tensor`): saliency map.
        size (bool or tuple of int): if a tuple (i.e., (width, height),
            resize :attr:`saliency` to :attr:`size`. If True, resize
            :attr:`saliency: to the shape of :attr:`tensor`; otherwise,
            return :attr:`saliency` unchanged.
        mode (str): mode for :func:`torch.nn.functional.interpolate`.

    Returns:
        :class:`torch.Tensor`: Resized saliency map.
    """
    if size is not False:
        if size is True:
            size = tensor.shape[2:]
        elif isinstance(size, tuple) or isinstance(size, list):
            # width, height -> height, width
            size = size[::-1]
        else:
            assert False, "resize must be True, False or a tuple."
        saliency = F.interpolate(
            saliency, size, mode=mode, align_corners=False)
    return saliency

In [ ]:
# Reference: https://github.com/jacobgil/pytorch-grad-cam/blob/master/tutorials/vision_transformers.md

def reshape_transform(tensor, height=16, width=16):
    result = tensor[1:, :  , :].reshape(tensor.size(1),
        height, width, tensor.size(2))

    result = result.transpose(2, 3).transpose(1, 2)

    return result

Determinism

In [ ]:
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

Results directory

In [ ]:
os.makedirs('./results', exist_ok = True) 

## Core Grad-CAM code

In [ ]:
def run_gradcam(dataset, image_path, category_id, text_to_add, save_input_path, save_gradcam_path):
        
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    model, preprocess = clip.load('ViT-L/14', device=device)
        
    image = preprocess(Image.open(image_path)).unsqueeze(0).to(device)
    
    if save_input_path is not None:
        invt = transforms.Compose([transforms.Normalize(mean = [ 0., 0., 0. ],
                                                         std = [1/0.26862954, 1/0.26130258, 1/0.27577711]),
                                   transforms.Normalize(mean = [-0.48145466, -0.4578275, -0.40821073],
                                                         std = [ 1., 1., 1. ]),
        ])
        image_denormalized = invt(image)
        save_image(image_denormalized[0], save_input_path)
        
    saliency_layer = get_module(model, 'visual.transformer.resblocks.11.ln_1')

    probe = Probe(saliency_layer, target='output')

    prompts = []
    if dataset == 'fish':
        for val in fish_classes:
            prompts.append('a photo of ' + val + ', a type of fish' + text_to_add)

    text = clip.tokenize(prompts).to(device)
    image_features = model.encode_image(image)
    text_features = model.encode_text(text)
    image_features_norm = image_features.norm(dim=-1, keepdim=True)
    image_features_new = image_features / image_features_norm
    text_features_norm = text_features.norm(dim=-1, keepdim=True)
    text_features_new = text_features / text_features_norm
    logit_scale = model.logit_scale.exp()
    logits_per_image = logit_scale * image_features_new @ text_features_new.t()

    probs = logits_per_image.softmax(dim=-1)
        
    probs[0, category_id].backward()
    
    probe_t = reshape_transform(probe.data[0], height = 16, width = 16)
    probe_t.grad = reshape_transform(probe.data[0].grad, height = 16, width = 16)
    
    saliency = gradient_to_grad_cam_saliency(probe_t)
            
    saliency = resize_saliency(image,
                               saliency,
                               size = True,
                               mode='bilinear')

    val_max = saliency.detach().cpu().numpy()[0, 0].max()
    
    heatmap = np.uint8(255 * saliency.detach().cpu().numpy()[0, 0] / val_max)
    
    if save_gradcam_path is not None:
        plt.imsave('/'.join(save_gradcam_path.split('/')[:-1]) + '/g_' + save_gradcam_path.split('/')[-1], heatmap)
    
    heatmap = cv2.applyColorMap(np.uint8(255 * saliency.detach().cpu().numpy()[0, 0] / val_max), cv2.COLORMAP_JET)
        
    if save_gradcam_path is not None:
        plt.imsave(save_gradcam_path, heatmap)

In [ ]:
def save_superimposed_img(gradcam_path, image_path, save_path, desc):
    heatmap = cv2.imread(gradcam_path)
    img = cv2.imread(image_path)
    heatmap = np.uint8(heatmap)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    superimposed_img = heatmap * 0.4 + img * 0.6
    cv2.imwrite(save_path, superimposed_img)
    image = cv2.imread(save_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    plt.figure()
    plt.title(desc)
    plt.imshow(image)

## Run Script

In [ ]:
def run(dataset, image_path, texts_to_add, class_nb):

    folder = './results/' + image_path.split('/')[-1].split('.')[0] + '/'
    os.makedirs(folder, exist_ok = True) 

    for i, text in enumerate(texts_to_add):
        run_gradcam(dataset = dataset, \
                    image_path = image_path, \
                    category_id = class_nb, \
                    text_to_add = text, \
                    save_input_path = folder + image_path.split('/')[-1].split('.')[0] + '_input.jpg', \
                    save_gradcam_path = folder + image_path.split('/')[-1].split('.')[0] + '_gradcam_desc_{}.jpg'.format(i))

        save_superimposed_img(gradcam_path = folder + image_path.split('/')[-1].split('.')[0] + '_gradcam_desc_{}.jpg'.format(i), \
                              image_path = folder + image_path.split('/')[-1].split('.')[0] + '_input.jpg', \
                              save_path = folder + image_path.split('/')[-1].split('.')[0] + '_superimposed_desc_{}.jpg'.format(i), \
                              desc = text)

## Results

In [ ]:
run(dataset = 'fish',
    image_path = './images/bangus.png',
    texts_to_add = ['.', ', in water.'],
    class_nb = fish_classes.index('Bangus'))